# C — Pareto Testing (baseline demo)
Pareto filter on empirical summaries, then test reliability objective with Holm.

In [ ]:
import numpy as np, pandas as pd
from utils.csvio import load_losses_csv
from utils.testing import pareto_mask, holm_bonferroni, hoeffding_pval

csv_path='data/sample_real_losses.csv'
alpha_risk=1.2
alpha_mtp=0.05

ids, L, cols = load_losses_csv(csv_path)
mean = L.mean(axis=1)
std  = L.std(axis=1)
vals = np.vstack([mean, std]).T  # two objectives (both minimize)

mask = pareto_mask(vals, minimize=True)
pareto_ids = ids[mask]

rows=[]
for i, hp in enumerate(ids):
    if hp not in set(pareto_ids): 
        continue
    losses=L[i]
    rhat=float(losses.mean()); n=len(losses)
    p=hoeffding_pval(rhat, n, alpha_risk)
    rows.append({'hyperparam_id':hp,'mean_loss':rhat,'pval':p})

df=pd.DataFrame(rows)
df['selected']=holm_bonferroni(df['pval'].values, alpha=alpha_mtp)
df.sort_values(['selected','mean_loss'], ascending=[False,True])